## Importing Libraries

In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE,ADASYN
from sklearn.ensemble import AdaBoostClassifier,RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,f1_score,make_scorer
from sklearn.tree import  DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV,cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from collections import Counter
import joblib
import warnings
import optuna

## Model's Object Names

In [55]:
warnings.filterwarnings('ignore')
scaler=StandardScaler()
smoteenn=SMOTEENN()
smote=SMOTE(random_state=42)
adaysn=ADASYN(random_state=42)
minmax=MinMaxScaler()
model_dt=DecisionTreeClassifier()
model_rf=RandomForestClassifier(n_estimators=50)
model_xgb=XGBClassifier(random_state=42)
model_rfp=RandomForestClassifier(class_weight='balanced',random_state=42,n_estimators=300,max_depth=6)
model_cat=CatBoostClassifier(auto_class_weights='Balanced',random_state=42,verbose=0)
model_xgb_hyper=XGBClassifier(random_state=42,eval_metric='logloss')

In [56]:
df=pd.read_csv('./dataset/Customer-Churn.csv')

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.Churn.value_counts()/len(df.Churn)*100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

In [6]:
y=df['Churn']
x=df.drop(columns=['Churn','customerID'])


In [7]:
print("shape of x is : ",x.shape)

shape of x is :  (7043, 19)


In [8]:
print("shape of y :",y.shape)

shape of y : (7043,)


In [9]:
x=pd.get_dummies(data=x,drop_first=True)
y=df['Churn'].map({"Yes":1,"No":0})
x

,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,TotalCharges_995.35,TotalCharges_996.45,TotalCharges_996.85,TotalCharges_996.95,TotalCharges_997.65,TotalCharges_997.75,TotalCharges_998.1,TotalCharges_999.45,TotalCharges_999.8,TotalCharges_999.9
0,0,1,29.85,False,True,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,34,56.95,True,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,2,53.85,True,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0,45,42.30,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,2,70.70,False,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,True,True,True,True,False,True,False,...,False,False,False,False,False,False,False,False,False,False
7039,0,72,103.20,False,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,False
7040,0,11,29.60,False,True,True,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
7041,1,4,74.40,True,True,False,True,False,True,True,...,False,False,False,False,False,False,False,False,False,False


In [10]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [11]:

model_dt.fit(x_train,y_train)
model_dt1_y_pred=model_dt.predict(x_test)
print(classification_report(y_test,model_dt1_y_pred))

              precision    recall  f1-score   support

           0       0.84      0.85      0.85      1058
           1       0.53      0.52      0.53       351

    accuracy                           0.77      1409
   macro avg       0.69      0.69      0.69      1409
weighted avg       0.77      0.77      0.77      1409



In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [13]:
df_copy=df.copy()

In [14]:
df_copy['TotalCharges']=pd.to_numeric(df_copy.TotalCharges,errors='coerce')

In [15]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [16]:
df_copy.loc[df_copy['TotalCharges'].isnull()==True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [17]:
df_copy.dropna(how='any',inplace=True)

In [18]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [19]:
df_copy.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [20]:
bins=[0,12,24,36,48,60,72]
labels=['1-12','13-24','25-36','37-48','49-60','61-72']
df_copy['tenure_bin']=pd.cut(df_copy['tenure'],bins=bins,labels=labels,include_lowest=True)


In [21]:
df_copy[['tenure','tenure_bin']].head(10)

,tenure,tenure_bin
0,1,1-12
1,34,25-36
2,2,1-12
3,45,37-48
4,2,1-12
5,8,1-12
6,22,13-24
7,10,1-12
8,28,25-36
9,62,61-72


In [22]:
df_copy['tenure_bin'].value_counts().sort_index()

tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
49-60     832
61-72    1407
Name: count, dtype: int64

In [23]:
df_copy.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [24]:
y=df_copy['Churn']
x=df_copy.drop(columns=['customerID','tenure','Churn'])

print("Shape of X is :",x.shape)
print("Shape of Y is :",y.shape)

Shape of X is : (7032, 19)
Shape of Y is : (7032,)


In [25]:
x=pd.get_dummies(x,drop_first=True)
y=df_copy['Churn'].map({'Yes':1,'No':0})


In [26]:
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_49-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,...,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,...,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,...,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,...,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,...,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,...,False,True,False,False,True,False,False,False,False,False


## Train Test Split

In [27]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Feature Scaling - StandardScaler

In [28]:
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

## DescionTree

In [29]:
model_dt.fit(x_train,y_train)
model_dt2_y_pred=model_dt.predict(x_test)
print(classification_report(y_test,model_dt2_y_pred))

              precision    recall  f1-score   support

           0       0.81      0.80      0.81      1024
           1       0.48      0.50      0.49       383

    accuracy                           0.72      1407
   macro avg       0.65      0.65      0.65      1407
weighted avg       0.72      0.72      0.72      1407



## Feature Scaling - MinMaxScaler

In [30]:
x_train_mns=minmax.fit_transform(x_train)
y_train_mns=minmax.transform(x_test)

## DescionTree With MIN MAX

In [31]:
model_dt.fit(x_train_mns,y_train)
model_dt3_y_pred=model_dt.predict(x_test)
print(classification_report(y_test,model_dt3_y_pred))

              precision    recall  f1-score   support

           0       0.81      0.64      0.72      1024
           1       0.38      0.60      0.47       383

    accuracy                           0.63      1407
   macro avg       0.60      0.62      0.59      1407
weighted avg       0.69      0.63      0.65      1407



## DescionTree With SMOTENN

In [32]:
x_train_smotenn,y_train_smotenn=smoteenn.fit_resample(x_train,y_train)
model_dt.fit(x_train_smotenn,y_train_smotenn)
model_dt4_y_pred=model_dt.predict(x_test)
print(classification_report(y_test,model_dt4_y_pred))

              precision    recall  f1-score   support

           0       0.88      0.71      0.79      1024
           1       0.49      0.74      0.59       383

    accuracy                           0.72      1407
   macro avg       0.68      0.73      0.69      1407
weighted avg       0.77      0.72      0.73      1407



## RandomForest With SMOTENN

In [33]:
model_rf.fit(x_train_smotenn,y_train_smotenn)
model_rf1_y_pred=model_rf.predict(x_test)
print(classification_report(y_test,model_rf1_y_pred))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1024
           1       0.51      0.78      0.62       383

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.79      0.73      0.75      1407



## Xgboost Without SMOTENN

In [34]:
model_xgb.fit(x_train,y_train)
model_xgb1_y_pred=model_xgb.predict(x_test)
print(classification_report(y_test,model_xgb1_y_pred))

              precision    recall  f1-score   support

           0       0.83      0.88      0.85      1024
           1       0.62      0.50      0.55       383

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



## Xgboost With SMOTENN

In [35]:
model_xgb.fit(x_train_smotenn,y_train_smotenn)
model_xgb2_y_pred=model_xgb.predict(x_test)
print(classification_report(y_test,model_xgb2_y_pred))

              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1024
           1       0.51      0.76      0.61       383

    accuracy                           0.73      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.73      0.75      1407



##  Xgboost With SMOTE

In [36]:
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)
print("Before smote",Counter(y_train))
print("After smote",Counter(y_train_smote))
model_xgb.fit(x_train_smote,y_train_smote)
model_xgb3_y_pred=model_xgb.predict(x_test)
print(classification_report(y_test,model_xgb3_y_pred))

Before smote Counter({0: 4139, 1: 1486})
After smote Counter({0: 4139, 1: 4139})
              precision    recall  f1-score   support

           0       0.84      0.85      0.84      1024
           1       0.58      0.56      0.57       383

    accuracy                           0.77      1407
   macro avg       0.71      0.70      0.71      1407
weighted avg       0.77      0.77      0.77      1407



## Xgboost With ADAYSN

In [37]:
x_train_adaysn,y_train_adaysn=adaysn.fit_resample(x_train,y_train)
print("Before adaysn : ",Counter(y_train))
print("After adaysn : ",Counter(y_train_adaysn))
model_xgb.fit(x_train_adaysn,y_train_adaysn)
model_xgb4_y_pred=model_xgb.predict(x_test)
print(accuracy_score(y_test,model_xgb4_y_pred))
print(classification_report(y_test,model_xgb4_y_pred))
print(confusion_matrix(y_test,model_xgb4_y_pred))

Before adaysn :  Counter({0: 4139, 1: 1486})
After adaysn :  Counter({0: 4139, 1: 4058})
0.7683013503909026
              precision    recall  f1-score   support

           0       0.83      0.85      0.84      1024
           1       0.58      0.55      0.56       383

    accuracy                           0.77      1407
   macro avg       0.71      0.70      0.70      1407
weighted avg       0.76      0.77      0.77      1407

[[872 152]
 [174 209]]


## Xgboost With Scale_Pos_Weight

In [38]:
scale_pos_weight=(y_train==0).sum()/(y_train==1).sum()
print("Scale Pos Weight is :",scale_pos_weight)
model_xgb_scale_poss_weight=XGBClassifier(scale_pos_weight=scale_pos_weight,eval_metric='logloss',random_state=42)
model_xgb_scale_poss_weight.fit(x_train,y_train)
model_xgb5_y_pred=model_xgb_scale_poss_weight.predict(x_test)
print(accuracy_score(y_test,model_xgb5_y_pred))
print(classification_report(y_test,model_xgb5_y_pred))
print(confusion_matrix(y_test,model_xgb5_y_pred))

Scale Pos Weight is : 2.785329744279946
0.757640369580668
              precision    recall  f1-score   support

           0       0.86      0.80      0.83      1024
           1       0.55      0.65      0.59       383

    accuracy                           0.76      1407
   macro avg       0.70      0.72      0.71      1407
weighted avg       0.77      0.76      0.76      1407

[[817 207]
 [134 249]]


## Random Forest With Parameters

In [39]:
model_rfp.fit(x_train,y_train)
model_rfp1_y_pred=model_rfp.predict(x_test)
print(classification_report(y_test,model_rfp1_y_pred))

              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1024
           1       0.51      0.78      0.61       383

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.75      1407



## CatBoost With Parameters

In [40]:
model_cat.fit(x_train,y_train)
model_cat_y_pred=model_cat.predict(x_test)
print(classification_report(y_test,model_cat_y_pred))

              precision    recall  f1-score   support

           0       0.88      0.78      0.82      1024
           1       0.54      0.71      0.62       383

    accuracy                           0.76      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.79      0.76      0.77      1407



## Light GBM With Scale_Pos_Weight

In [41]:
model_lgbm=LGBMClassifier(random_state=42,scale_pos_weight=scale_pos_weight)
model_lgbm.fit(x_train,y_train)
model_lgbm1_y_pred=model_lgbm.predict(x_test)
print(classification_report(y_test,model_lgbm1_y_pred))

[LightGBM] [Info] Number of positive: 1486, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.264178 -> initscore=-1.024366
[LightGBM] [Info] Start training from score -1.024366
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1024
           1       0.54      0.73      0.62       383

    accuracy                           0.76      1407
   macro avg       0.71      0.75      0.72      1407
weighted avg       0.79      0.76      0.77      1407



## Adaboost with Sample Weight

In [42]:
negative_count=(y_train==0).sum()
positive_count=(y_train==1).sum()
np_weight=negative_count/positive_count

sample_weight=np.where(y_train==1,np_weight,1.0)

model_ada_weighted=AdaBoostClassifier(n_estimators=50,random_state=42)

model_ada_weighted.fit(x_train,y_train,sample_weight=sample_weight)

y_pred_ada_weighted=model_ada_weighted.predict(x_test)


print("Accuracy Score is : ",accuracy_score(y_pred_ada_weighted,y_test))
print("Classification Report is : \n",classification_report(y_pred_ada_weighted,y_test))
print("Confusion Matrix \n",confusion_matrix(y_pred_ada_weighted,y_test))

Accuracy Score is :  0.746268656716418
Classification Report is : 
               precision    recall  f1-score   support

           0       0.74      0.89      0.81       849
           1       0.76      0.52      0.62       558

    accuracy                           0.75      1407
   macro avg       0.75      0.71      0.72      1407
weighted avg       0.75      0.75      0.73      1407

Confusion Matrix 
 [[758  91]
 [266 292]]


In [43]:
param_grid={
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]  # try both balanced and weighted
}
random_search=RandomizedSearchCV(model_xgb_hyper,param_grid,random_state=42,scoring='f1',n_jobs=-1,cv=5,n_iter=50)

random_search.fit(x_train,y_train)


print("Best Parameters : ",random_search.best_params_)
print("best f1 score is : ",random_search.best_score_)

best_model=random_search.best_estimator_
y_pred_best=best_model.predict(x_test)

print("\n Test Classifiction Report \n ",classification_report(y_pred_best,y_test))

Best Parameters :  {'subsample': 0.8, 'scale_pos_weight': np.float64(2.785329744279946), 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.8}
best f1 score is :  0.6277556973527989

 Test Classifiction Report 
                precision    recall  f1-score   support

           0       0.74      0.89      0.81       852
           1       0.76      0.52      0.62       555

    accuracy                           0.75      1407
   macro avg       0.75      0.71      0.72      1407
weighted avg       0.75      0.75      0.74      1407



In [44]:
joblib.dump(best_model,"./models/best_xgboost_model.pkl")
print("Model is Saved Succesfully")

Model is Saved Succesfully


In [50]:
def objective(trial):
    params={
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2), 
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    model=XGBClassifier(**params)

    f1_scoreer=make_scorer(f1_score,average='binary',pos_label=1)
    score=cross_val_score(model,x_train,y_train,cv=5,scoring=f1_scoreer,n_jobs=-1).mean()

    return score
study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=100)

print("Best params are ",study.best_params)
print("best score is :",study.best_value)

best_optuna_model=XGBClassifier(**study.best_params)
best_optuna_model.fit(x_train,y_train)
y_pred_optuna=best_optuna_model.predict(x_test)
print("classification_report is \n : ",classification_report(y_test,y_pred_optuna))

[I 2026-03-06 22:45:30,144] A new study created in memory with name: no-name-5bceb019-ae6c-48ce-a1af-b087977b91f8
[I 2026-03-06 22:45:31,154] Trial 0 finished with value: 0.6192958691423083 and parameters: {'max_depth': 3, 'learning_rate': 0.055032685642513866, 'n_estimators': 902, 'subsample': 0.8319135686809065, 'colsample_bytree': 0.6609391728804548, 'min_child_weight': 7, 'gamma': 0.37244775674427766, 'reg_alpha': 0.3215244399205798, 'reg_lambda': 1.95632699457203, 'scale_pos_weight': np.float64(2.785329744279946)}. Best is trial 0 with value: 0.6192958691423083.
[I 2026-03-06 22:45:32,083] Trial 1 finished with value: 0.6082500971212775 and parameters: {'max_depth': 6, 'learning_rate': 0.062227253828817164, 'n_estimators': 639, 'subsample': 0.9450701768080566, 'colsample_bytree': 0.9910818289284338, 'min_child_weight': 9, 'gamma': 0.46428923831163615, 'reg_alpha': 0.11113608043043055, 'reg_lambda': 0.32493526910338644, 'scale_pos_weight': np.float64(2.785329744279946)}. Best is tr

Best params are  {'max_depth': 6, 'learning_rate': 0.022115541143650032, 'n_estimators': 219, 'subsample': 0.6295470144507495, 'colsample_bytree': 0.7718132768896089, 'min_child_weight': 7, 'gamma': 0.32981335098363035, 'reg_alpha': 0.2981902127645493, 'reg_lambda': 0.12235204310395453, 'scale_pos_weight': np.float64(2.785329744279946)}
best score is : 0.6316919246298747
classification_report is 
 :                precision    recall  f1-score   support

           0       0.89      0.76      0.82      1024
           1       0.54      0.75      0.63       383

    accuracy                           0.76      1407
   macro avg       0.71      0.75      0.72      1407
weighted avg       0.79      0.76      0.77      1407



In [57]:
joblib.dump(best_optuna_model,'./models/best_optuna_model.pkl')
print("optuna model saved Successfully")

optuna model saved Successfully


In [58]:
joblib.dump(model_ada_weighted,"./models/adaboost_weighted_model.pkl")
print("adaboost weighted model saved Successfully")

adaboost weighted model saved Successfully
